#### 1、什么是Agent

**Agent（智能体）**是具有特定目标和能力的AI助手，可以：

- 理解用户意图
- 规划任务步骤
- 调用工具执行
- 做出决策
- 与其他智能体协作



#### 2、为什么要用Agent

传统的大模型就像一位聪明的顾问：它能回答问题、给出建议，但**不会自己动手做事**。而Agent是让AI真正"动起来"的组件，**让AI成为"执行者"，而不仅仅是"建议者"**，从而突破大模型的三大局限知识时效性、操作能力和数据访问限制。同时Agent能分解复杂任务、管理多轮对话、记忆历史交互，实现从单次问答到持续协作。



#### 3、如何使用Agent

在 OpenAI Agents SDK 中，使用一个 Agent 并让它真正“动起来”，需要完成两个核心动作：

1. **配置模型客户端**（告诉 Agent 用哪个大模型）
2. **运行 Agent 并获取结果**

>  特别注意：
>  如果你使用的是 **OpenAI 官方 API**，可以直接用 `model="gpt-4o"`；
>  但如果你使用的是 **代理节点**（如阿里通义、智谱、DeepSeek 等兼容 OpenAI 接口的模型），**必须先手动创建模型客户端**！
>
>  官网链接：https://github.com/openai/openai-agents-python

下面我们将从**最通用的场景出发**——使用国内兼容 OpenAI 的模型（如通义千问 Qwen）——来讲解三种主流的 Agent 使用方式。

 **为什么需要“实例化客户端”？**

传统大模型调用：

```
response = openai.chat.completions.create(model="gpt-4o", ...)
```

但在多 Agent 系统中，Agent 需要**自主决定何时调用模型、调用多少次**。因此 SDK 要求你提前提供一个**可复用的模型客户端对象**（`AsyncOpenAI`），而不是每次临时拼参数。

 所有非官方 OpenAI 模型（包括通过代理访问的 gpt-4o）都必须这样做！

官网链接：https://openai.github.io/openai-agents-python/models/

####  3.1. 三种方式配置模型客户端

（按作用范围排序）

| 方式                                            | 作用范围                    | 适用场景                  |
| ----------------------------------------------- | --------------------------- | ------------------------- |
| **方式一`set_default_openai_client`：全局设置** | 所有 Agent 共享同一个客户端 | 快速原型、单模型项目      |
| **方式二`ModelProvider`：运行时传入**           | 单次 Run 指定客户端         | 多模型混合调度            |
| **方式三`Agent.model`：绑定到单个 Agent**       | 仅该 Agent 使用指定客户端   | 精细控制每个 Agent 的模型 |



##### 方式一：全局设置
适用于整个项目只用一种模型的情况（如全部用 Qwen-Plus）

In [2]:
from openai import AsyncOpenAI

from agents import (
    Agent,
    Runner,
    set_default_openai_api,
    set_default_openai_client,
    set_tracing_disabled,
)

BASE_URL ="https://dashscope.aliyuncs.com/compatible-mode/v1"
API_KEY = "sk-037fa48f3ac5454d9f4be3f75368a383"
MODEL_NAME = "qwen3-max"

# 1. 创建异步客户端（必须是 AsyncOpenAI！）
client = AsyncOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)
# 2. 全局注册客户端（所有 Agent 都会用它）a
set_default_openai_client(client=client)


set_default_openai_api("chat_completions")   # 必须指定为 chat_completions
set_tracing_disabled(disabled=True)  # 避免因 tracing 导致 401 错误


async def main():
    # 3. 创建 Agent（只需指定 model 名称）
    agent = Agent(
        name="文学专家", # Agent的名字
        instructions="你只会用七言绝句回应", # 给Agent的指令
        model=MODEL_NAME,  # 这里只是字符串
    )
    # 3.运行Aagent
    result = await Runner.run(agent, "给我写一首关于春天的七言绝句") # 运行Agent

    # 4.得到Agent结果
    print(result.final_output)  # 得到Agent结果

await main()


春江水暖柳丝长，  
桃李争妍映日光。  
燕语呢喃花影里，  
东风又绿旧池塘。


注意事项：
- 必须调用 `set_default_openai_api("chat_completions")`，否则报 400
- 必须调用 `set_tracing_disabled(True)`，否则可能因 tracing 服务不可用报 401
- 客户端必须是 `AsyncOpenAI`（即使你后面用同步调用）

##### 方式二：运行时传入（灵活调度多模型）
需要在开发/生产环境切换模型、或者构建一个平台让用户自己选择模型来运行同一个 Agent

In [3]:
from __future__ import annotations

import asyncio

from openai import AsyncOpenAI

from agents import (
    Agent,
    Model,
    ModelProvider,
    OpenAIChatCompletionsModel,
    RunConfig,
    Runner,
    function_tool,
    set_tracing_disabled,
)

BASE_URL =  "https://dashscope.aliyuncs.com/compatible-mode/v1"
API_KEY =  "sk-037fa48f3ac5454d9f4be3f75368a383"
MODEL_NAME =  "qwen-plus"

client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)
set_tracing_disabled(disabled=True)

# 1. 自定义ModelProvider类，实现ModelProvider接口的get_model方法
class CustomModelProvider(ModelProvider):

     # 实现ModelProvider接口的get_model方法
    def get_model(self, model_name: str) -> Model:
        print(model_name)

        return OpenAIChatCompletionsModel(model= model_name, openai_client=client) # 直接返回OpenAIChatCompletionsModel  不用在调用set_default_openai_api("chat_completions")设置默认api为chat_completions


# 2. 创建CustomModelProvider实例
CUSTOM_MODEL_PROVIDER = CustomModelProvider()

async def main():
    # 3. 使用CustomModelProvider实例创建Agent
    agent = Agent(name="Assistant", instructions="你只会用七言绝句回应.",model=MODEL_NAME) # 字符串，由 provider 解析

    # 4. 运行Agent
    result = await Runner.run(
        agent,
        input="给我写一首关于春天的七言绝句",
        run_config=RunConfig(model_provider=CUSTOM_MODEL_PROVIDER), # 在 run 时传入自定义 provider

    )
    # 5. 获取Agent运行结果
    print(result.final_output)


await main()

qwen-plus
《春晓即事》  
风梳柳线蘸溪青，燕剪云痕过短亭。  
一树梨花吹作雪，半山新绿染空灵。


优点：无需全局污染，每个 Run 可独立配置模型来源

##### 方式三：绑定到单个 Agent（最精细控制）
适用于不同 Agent 使用不同模型（如客服用 Qwen，分析用 DeepSeek）


In [4]:
import asyncio


from openai import AsyncOpenAI

from agents import Agent, OpenAIChatCompletionsModel, Runner, function_tool, set_tracing_disabled

BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
API_KEY =  "sk-037fa48f3ac5454d9f4be3f75368a383"
MODEL_NAME = "qwen-plus"

# 1. 创建AsyncOpenAI客户端实例
client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)
set_tracing_disabled(disabled=True)

async def main():
    # 2. 创建Agent实例
    agent = Agent(
        name="Assistant",
        instructions="你只会用七言绝句回应.",
        model=OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=client), # 3.直接传入模型对象（不再是字符串！）OpenAIChatCompletionsModel
    )

    # 4. 运行Agent
    result = await Runner.run(agent, "给我写一首关于春天的七言绝句")

    # 5. 获取Agent运行结果
    print(result.final_output)



await  main()

《春晓即事》  
风梳柳线蘸溪青，燕剪云痕过短亭。  
一树梨花吹作雪，半山新绿染空灵。


优点：Agent 自包含，不依赖外部配置

#### 2. 三种调用方式

无论用哪种模型配置方式，Agent 都支持以下三种调用模式：

官网链接：https://openai.github.io/openai-agents-python/running_agents/

| 调用方式     | 方法                    | 返回类型             | 适用场景           |
| ------------ | ----------------------- | -------------------- | ------------------ |
| **同步调用** | `Runner.run_sync()`     | `RunResult`          | 脚本、简单后端     |
| **异步调用** | `await Runner.run()`    | `RunResult`          | FastAPI、高并发    |
| **流式调用** | `Runner.run_streamed()` | `RunResultStreaming` | 实时对话、SSE 接口 |

##### 示例：同步调用（最简单）

In [ ]:
result = Runner.run_sync(agent, "问题")
print(result.final_output)

##### 示例：异步调用（推荐 Web 后端）

In [ ]:
result = await Runner.run(agent, "问题")

##### 示例：流式调用（实现“打字机效果”）

In [ ]:
streaming_result = Runner.run_streamed(agent, "问题")

async for event in streaming_result.stream_events():
    if event.type == "raw_response_event":
        delta = event.data.delta  # 增量文本
        print(delta, end="", flush=True)

# 最终完整结果仍可通过 final_output 获取
print("\n完整答案：", streaming_result.final_output)

流式事件类型说明（）：

- `"tool_call"`：Agent 准备调用工具
- `"tool_output"`：工具返回结果
- `"raw_response_event"`：模型生成的文本增量（含思考过程）
- `"final_answer"`：最终回答完成

#### 3. tool集成Agent

In [12]:
import asyncio


from openai import AsyncOpenAI

from agents import Agent, OpenAIChatCompletionsModel, Runner, function_tool, set_tracing_disabled

BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
API_KEY =  "sk-26d57c968c364e7bb14f1fc350d4bff0"
MODEL_NAME = "qwen-plus"

client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)
set_tracing_disabled(disabled=True)


@function_tool
def get_weather(city: str) -> str:
    """获取指定城市的天气信息"""
    return f"天气信息: {city} 是晴天"

async def main():
    agent = Agent(
        name="天气助手",
        instructions="你是一个天气助手，你只能回答关于天气的问题。",
        model=OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=client),
        tools=[get_weather],

    )
    result = await Runner.run(agent, "武汉的天气")
    print(result.final_output)


await main()

武汉今天是晴天，天气晴朗，适合外出活动。


注意事项：

- 函数上使用@function_tool注解
- tools必须是列表

#### 4. Agent输出结构化对象

In [18]:
import asyncio
import json
from openai import AsyncOpenAI
from agents import Agent, OpenAIChatCompletionsModel, Runner, function_tool, set_tracing_disabled
from pydantic import BaseModel

BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
API_KEY =  "sk-26d57c968c364e7bb14f1fc350d4bff0"
MODEL_NAME = "qwen-plus"

client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)
set_tracing_disabled(disabled=True)

class WeatherResult(BaseModel):
    city: str
    condition: str
    source: str
    message: str

@function_tool
def get_weather(city: str) -> str:
    """根据城市获取城市的天气"""
    return json.dumps(
        {"city": city, "condition": "晴", "source": "tool", "message": f"{city}天气晴朗"},
        ensure_ascii=False,
    )

async def main():
    agent = Agent(
        name="天气助手",
        instructions=(
            "你是天气助手，只回答天气问题。"
            "当用户问天气时必须调用 get_weather。"
        ),
        model=OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=client),
        tools=[get_weather],
        output_type=WeatherResult,
    )

    result = await Runner.run(agent, "武汉的天气")
    # result.final_output 不再是 str，而是 WeatherResult（或可当作 dict 使用）
    print("final_output:", result.final_output)

    # 如果你想转 dict

    print("as dict:", type(result.final_output.model_dump()))

await main()


final_output: city='武汉' condition='晴' source='weather_api' message='今天天气晴朗，气温20度。'
as dict: <class 'dict'>


#### 5、Agent使用场景

| 场景     | Agent类型 | 职责                   |
| :------- | :-------- | :--------------------- |
| 客服系统 | 客服专员  | 解答常见问题           |
| 技术支持 | 技术专家  | 解决技术问题           |
| 销售咨询 | 销售顾问  | 产品推荐和报价         |
| 订单处理 | 订单专员  | 处理订单相关事务       |
| 路由分发 | 路由助手  | 将问题分发给合适的专家 |